In [ ]:
!wget -q "https://www.timeseriesclassification.com/aeon-toolkit/Heartbeat.zip"
!unzip -q Heartbeat.zip

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import time

# Load
train = pd.read_csv("Heartbeat_TRAIN.txt", header=None, sep=r'\s+')
test  = pd.read_csv("Heartbeat_TEST.txt",  header=None, sep=r'\s+')

X_train = train.iloc[:, 1:].values.astype(np.float32)
y_train = train.iloc[:, 0].values
X_test  = test.iloc[:, 1:].values.astype(np.float32)
y_test  = test.iloc[:, 0].values

le = LabelEncoder()
y_train = le.fit_transform(y_train).astype(np.int64)
y_test  = le.transform(y_test).astype(np.int64)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Sequence length: {X_train.shape[1]}")
print(f"Classes: {len(np.unique(y_train))}")

In [ ]:
# Normalize per sample
mean = X_train.mean(axis=1, keepdims=True)
std  = X_train.std(axis=1, keepdims=True) + 1e-8
X_train = (X_train - mean) / std

mean = X_test.mean(axis=1, keepdims=True)
std  = X_test.std(axis=1, keepdims=True) + 1e-8
X_test = (X_test - mean) / std

# Add channel dimension
X_train_t = torch.tensor(X_train).unsqueeze(1)
X_test_t  = torch.tensor(X_test).unsqueeze(1)
y_train_t = torch.tensor(y_train)
y_test_t  = torch.tensor(y_test)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=32, shuffle=False)

print(f"Input shape: {X_train_t.shape}")

In [ ]:
class CNN_TemporalPooling(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.network(x)
        x = self.gap(x).squeeze(-1)
        return self.classifier(x)


class CNN_LSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.lstm = nn.LSTM(input_size=128, hidden_size=128,
                            num_layers=2, batch_first=True)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        _, (h_n, _) = self.lstm(x)
        return self.classifier(h_n[-1])

In [ ]:
def train_model_es(model, train_loader, test_loader, epochs=50, lr=1e-3, patience=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_losses, test_accs = [], []
    best_acc = 0.0
    patience_counter = 0
    best_epoch = 0
    start = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                pred = model(X_batch.to(device)).argmax(dim=1).cpu().numpy()
                preds.extend(pred)
                labels.extend(y_batch.numpy())

        acc = accuracy_score(labels, preds)
        train_losses.append(epoch_loss / len(train_loader))
        test_accs.append(acc)

        # Early stopping on accuracy
        if acc > best_acc:
            best_acc = acc
            patience_counter = 0
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'best_model.pt')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}, best epoch was {best_epoch}")
                break

        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1} | Loss: {train_losses[-1]:.4f} | Acc: {acc:.4f}")

    training_time = time.time() - start
    return train_losses, test_accs, training_time, best_acc, best_epoch

In [ ]:
import pickle

NUM_CLASSES = len(np.unique(y_train))

set_seed(42)
print("Training CNN + Temporal Pooling...")
pool_model = CNN_TemporalPooling(NUM_CLASSES)
pool_losses, pool_accs, pool_time, pool_best, pool_epoch = train_model_es(
    pool_model, train_loader, test_loader)

set_seed(42)
print("\nTraining CNN + LSTM...")
lstm_model = CNN_LSTM(NUM_CLASSES)
lstm_losses, lstm_accs, lstm_time, lstm_best, lstm_epoch = train_model_es(
    lstm_model, train_loader, test_loader)

# Save immediately
with open('heartbeat_curves.pkl', 'wb') as f:
    pickle.dump({
        'pool_losses': pool_losses, 'lstm_losses': lstm_losses,
        'pool_accs': pool_accs,     'lstm_accs': lstm_accs,
    }, f)

print(f"\n--- Final Results on Heartbeat (T=405) ---")
print(f"CNN+Pooling | Acc: {pool_best:.4f} | Stopped: epoch {pool_epoch} | Time: {pool_time:.1f}s")
print(f"CNN+LSTM    | Acc: {lstm_best:.4f} | Stopped: epoch {lstm_epoch} | Time: {lstm_time:.1f}s")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(pool_losses, label='CNN+Pooling')
axes[0].plot(lstm_losses, label='CNN+LSTM')
axes[0].set_title('Heartbeat — Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(pool_accs, label='CNN+Pooling')
axes[1].plot(lstm_accs, label='CNN+LSTM')
axes[1].set_title('Heartbeat — Test Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()